In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 17


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 3.220177970826626
Epoch 2/100, Loss: 3.2946626096963882
Epoch 3/100, Loss: 2.9108799025416374
Epoch 4/100, Loss: 2.780594825744629
Epoch 5/100, Loss: 2.7089645341038704
Epoch 6/100, Loss: 2.9518554657697678
Epoch 7/100, Loss: 3.169661320745945
Epoch 8/100, Loss: 3.3719179034233093
Epoch 9/100, Loss: 3.358076274394989
Epoch 10/100, Loss: 2.9740401282906532
Epoch 11/100, Loss: 3.104072652757168
Epoch 12/100, Loss: 2.5323802679777145
Epoch 13/100, Loss: 3.042662225663662
Epoch 14/100, Loss: 3.1814008578658104
Epoch 15/100, Loss: 2.920693077147007
Epoch 16/100, Loss: 3.4273706898093224
Epoch 17/100, Loss: 3.394430197775364


Epoch 18/100, Loss: 3.2393247708678246
Epoch 19/100, Loss: 2.944789670407772
Epoch 20/100, Loss: 2.5040613785386086
Epoch 21/100, Loss: 2.9604366049170494
Epoch 22/100, Loss: 2.606214888393879
Epoch 23/100, Loss: 2.6415571123361588
Epoch 24/100, Loss: 2.9473049342632294
Epoch 25/100, Loss: 3.466454356908798
Epoch 26/100, Loss: 3.181276135146618
Epoch 27/100, Loss: 2.8558930307626724
Epoch 28/100, Loss: 3.3609613478183746
Epoch 29/100, Loss: 2.836013026535511
Epoch 30/100, Loss: 2.7633543387055397
Epoch 31/100, Loss: 3.234404109418392


Epoch 32/100, Loss: 2.9210914447903633
Epoch 33/100, Loss: 2.99090788513422
Epoch 34/100, Loss: 3.2359130531549454
Epoch 35/100, Loss: 3.137990318238735
Epoch 36/100, Loss: 3.1652712374925613
Epoch 37/100, Loss: 2.8790626749396324
Epoch 38/100, Loss: 2.9359904527664185
Epoch 39/100, Loss: 3.3936280384659767
Epoch 40/100, Loss: 3.0210972279310226
Epoch 41/100, Loss: 3.3107111752033234
Epoch 42/100, Loss: 3.104798302054405
Epoch 43/100, Loss: 2.465897612273693
Epoch 44/100, Loss: 3.4011047184467316
Epoch 45/100, Loss: 2.687566392123699
Epoch 46/100, Loss: 3.0588990598917007
Epoch 47/100, Loss: 2.5144796296954155
Epoch 48/100, Loss: 2.899511307477951
Epoch 49/100, Loss: 2.8127253502607346


Epoch 50/100, Loss: 2.911402516067028
Epoch 51/100, Loss: 2.5655637234449387
Epoch 52/100, Loss: 2.82315231859684
Epoch 53/100, Loss: 2.983627274632454
Epoch 54/100, Loss: 3.124625600874424
Epoch 55/100, Loss: 2.8433491960167885
Epoch 56/100, Loss: 2.9473506063222885
Epoch 57/100, Loss: 2.700312338769436
Epoch 58/100, Loss: 2.56844948977232
Epoch 59/100, Loss: 2.956121362745762
Epoch 60/100, Loss: 3.1195111498236656
Epoch 61/100, Loss: 3.205742746591568
Epoch 62/100, Loss: 3.174479231238365
Epoch 63/100, Loss: 3.101216897368431
Epoch 64/100, Loss: 3.285276748239994
Epoch 65/100, Loss: 3.0550319626927376
Epoch 66/100, Loss: 2.940324991941452
Epoch 67/100, Loss: 3.0022691786289215


Epoch 68/100, Loss: 3.320753738284111
Epoch 69/100, Loss: 3.0609924718737602
Epoch 70/100, Loss: 2.500774271786213
Epoch 71/100, Loss: 2.8480304479599
Epoch 72/100, Loss: 3.8760014101862907
Epoch 73/100, Loss: 3.10883579403162
Epoch 74/100, Loss: 2.919016696512699
Epoch 75/100, Loss: 2.702549748122692
Epoch 76/100, Loss: 3.026038996875286
Epoch 77/100, Loss: 3.188251718878746
Epoch 78/100, Loss: 2.4111677706241608
Epoch 79/100, Loss: 2.977522023022175
Epoch 80/100, Loss: 3.2659203857183456
Epoch 81/100, Loss: 2.829516664147377
Epoch 82/100, Loss: 3.0797249525785446
Epoch 83/100, Loss: 3.3872137367725372
Epoch 84/100, Loss: 3.2675506621599197
Epoch 85/100, Loss: 3.0307698994874954


...

Epoch 24/100, Loss: 1.4215452745556831
Epoch 25/100, Loss: 1.4467428177595139
Epoch 26/100, Loss: 1.34921046346426
Epoch 27/100, Loss: 1.4102842435240746
Epoch 28/100, Loss: 1.3538193702697754
Epoch 29/100, Loss: 1.4898586459457874
Epoch 30/100, Loss: 1.2247243449091911
Epoch 31/100, Loss: 1.508617028594017
Epoch 32/100, Loss: 1.4380981996655464
Epoch 33/100, Loss: 1.4394028522074223
Epoch 34/100, Loss: 1.4834838807582855
Epoch 35/100, Loss: 1.4838044941425323
Epoch 36/100, Loss: 1.1732566840946674
Epoch 37/100, Loss: 1.268317885696888
Epoch 38/100, Loss: 1.3243778012692928
Epoch 39/100, Loss: 1.3397939652204514
Epoch 40/100, Loss: 1.4356773681938648


Epoch 41/100, Loss: 1.3764756880700588
Epoch 42/100, Loss: 1.2187106274068356
Epoch 43/100, Loss: 1.334678042680025
Epoch 44/100, Loss: 1.35208261013031
Epoch 45/100, Loss: 1.520542424172163
Epoch 46/100, Loss: 1.3606662042438984
Epoch 47/100, Loss: 1.301673948764801
Epoch 48/100, Loss: 1.4179662577807903
Epoch 49/100, Loss: 1.3128732778131962
Epoch 50/100, Loss: 1.4759175851941109
Epoch 51/100, Loss: 1.2880827337503433
Epoch 52/100, Loss: 1.3598748818039894
Epoch 53/100, Loss: 1.3681999817490578
Epoch 54/100, Loss: 1.2023664005100727
Epoch 55/100, Loss: 1.2856913655996323


Epoch 56/100, Loss: 1.5636460967361927
Epoch 57/100, Loss: 1.275397527962923
Epoch 58/100, Loss: 1.3364901058375835
Epoch 59/100, Loss: 1.208368968218565
Epoch 60/100, Loss: 1.5290706753730774
Epoch 61/100, Loss: 1.2076259665191174
Epoch 62/100, Loss: 1.402006808668375
Epoch 63/100, Loss: 1.3145335912704468
Epoch 64/100, Loss: 1.2569491304457188
Epoch 65/100, Loss: 1.3656842857599258
Epoch 66/100, Loss: 1.2562622055411339
Epoch 67/100, Loss: 1.3815204165875912
Epoch 68/100, Loss: 1.327040009200573
Epoch 69/100, Loss: 1.5080317668616772
Epoch 70/100, Loss: 1.1909778974950314


Epoch 71/100, Loss: 1.3399258963763714
Epoch 72/100, Loss: 1.4150159172713757
Epoch 73/100, Loss: 1.3533318750560284
Epoch 74/100, Loss: 1.3724414259195328
Epoch 75/100, Loss: 1.2116827890276909
Epoch 76/100, Loss: 1.400583241134882
Epoch 77/100, Loss: 1.5045307651162148
Epoch 78/100, Loss: 1.318667646497488
Epoch 79/100, Loss: 1.3797575123608112
Epoch 80/100, Loss: 1.3554354943335056
Epoch 81/100, Loss: 1.1777246333658695
Epoch 82/100, Loss: 1.3635064475238323
Epoch 83/100, Loss: 1.247484840452671
Epoch 84/100, Loss: 1.4268770925700665
Epoch 85/100, Loss: 1.384051401168108


Epoch 86/100, Loss: 1.3538228869438171
Epoch 87/100, Loss: 1.245988130569458
Epoch 88/100, Loss: 1.341013964265585
Epoch 89/100, Loss: 1.230606246739626
Epoch 90/100, Loss: 1.4472165927290916
Epoch 91/100, Loss: 1.594719361513853
Epoch 92/100, Loss: 1.3758226595818996
Epoch 93/100, Loss: 1.205508302897215
Epoch 94/100, Loss: 1.5281909219920635
Epoch 95/100, Loss: 1.3451565317809582
Epoch 96/100, Loss: 1.382330697029829
Epoch 97/100, Loss: 1.3709632977843285
Epoch 98/100, Loss: 1.4355743415653706
Epoch 99/100, Loss: 1.3645697683095932
Epoch 100/100, Loss: 1.236259564757347
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5806
